# 01｜資料品質檢查與分析資料準備

## 專案名稱

**半導體蝕刻製程智慧分析與虛擬量測**  
Semiconductor Etch Process Intelligence & Virtual Metrology

---

## 這份 Notebook 要做什麼？

00 已經把資料結構整理清楚：目前有 10 個 Lots、96 片 Process Wafers，其中 88 片同時有 89-point Measurement，而且 96 片 Wafer 共同擁有 31 個 Process Features。

所以這一份不再重新介紹資料，而是要回答一個更實際的問題：

> **這些資料在拿去做 Wafer Quality、Process EDA 和 Virtual Metrology 之前，有哪些品質問題需要先處理？**

我會依序檢查：

1. Measurement 的缺失值到底是什麼性質，資料提供者有沒有已經做過重建。
2. 有沒有重複資料或重複的 Wafer + XY 位置。
3. 88 片 Wafer 是否真的都使用同一組 89 個空間位置。
4. `oxide_etch`、`si_etch` 這些衍生欄位有沒有算對。
5. 數值有沒有最基本的不合理情況。
6. Lot metadata 的日期與 Conditioning 格式能不能整理成一致格式。
7. Process Data 有沒有 NaN、Inf 或完全不變的欄位。
8. Process timestamp 的 0.2 秒取樣、large gap 和 metadata 到底該怎麼解讀。
9. 最後把確認過的資料與 QC 規則輸出到 `data/interim/`，讓後面的 Notebook 不用各自再清一次。

我不會修改 `data/raw/`，也不會在這裡做模型。


## 1. 載入套件與設定資料位置

這一份要同時檢查 CSV、Excel 和 NetCDF，所以先把資料處理、正規表示式和 NetCDF 會用到的套件載入。


In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

from netCDF4 import Dataset, num2date

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 150)

print("套件載入完成")

套件載入完成


**這個結果代表什麼？**

顯示「套件載入完成」代表目前的環境可以正常開始做 Measurement、Excel 和 Process Data 的品質檢查。

這裡還沒有產生任何分析結論，只是先確認工具都能用。


### 1.1 設定 Raw Data 和 Interim Data 路徑

Raw Data 我不會改動，所有新建立的品質標記、整理後資料和 QC summary 都只會寫到 `data/interim/`。

另外先把 Process large gap 的門檻定成 **1.0 秒**。正常取樣大約是 0.2 秒，所以 >1 秒代表中間跨過超過 5 個一般 sampling intervals。

這裡先把它當成 QC flag，不直接等於「資料缺失」或「製程異常」。


In [2]:
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DATA_DIR = PROJECT_ROOT / "data" / "interim"

measurement_path = RAW_DATA_DIR / "Si_Oxide_etch_89_points.csv"
lot_status_path = RAW_DATA_DIR / "Lot_status.xlsx"
process_path = RAW_DATA_DIR / "Process_data.nc"
dictionary_path = RAW_DATA_DIR / "Dictionary_process.nc"

LARGE_GAP_THRESHOLD_SECONDS = 1.0

print("專案根目錄：", PROJECT_ROOT)
print("Raw Data：", RAW_DATA_DIR)
print("Interim Data：", INTERIM_DATA_DIR)

專案根目錄： C:\Users\momo8\Desktop\semiconductor-etch-process-intelligence
Raw Data： C:\Users\momo8\Desktop\semiconductor-etch-process-intelligence\data\raw
Interim Data： C:\Users\momo8\Desktop\semiconductor-etch-process-intelligence\data\interim


**這個結果代表什麼？**

三個路徑都指向同一個專案：

- Project root
- `data/raw/`
- `data/interim/`

所以後面讀的是原始資料，而新輸出的檔案只會進 `data/interim/`，不會覆蓋 Raw Data。


### 1.2 從 Raw Data 重新讀一次 01 需要的資料

01 要逐列檢查 Measurement，也要重新整理 `Lot_status.xlsx`，所以我不沿用 00 記憶體裡的變數，而是直接從原始檔重新讀。

這樣就算單獨重跑 01，也能確定是從沒有被修改過的 Raw Data 開始。


In [3]:
measurement_df = pd.read_csv(measurement_path)

lot_status_raw = pd.read_excel(
    lot_status_path,
    sheet_name="Tabelle1",
    header=None,
)

print("Measurement Data：", measurement_df.shape)
print("Lot Status Raw：", lot_status_raw.shape)

Measurement Data： (7832, 11)
Lot Status Raw： (36, 5)


**這個結果代表什麼？**

重新讀進來後：

- Measurement = **7,832 × 11**
- Lot Status Raw = **36 × 5**

和 00 看到的資料結構一致，所以 01 沒有讀到不同版本的檔案。


## 2. Measurement Missing Values 和 Interpolation

Measurement Data 最需要先釐清的是 `postox_thickness_nan` 和 `postox_thickness`。

這兩欄不是重複資料：

- `postox_thickness_nan`：保留原始 post-etch fitting failure，所以會有 NaN。
- `postox_thickness`：資料提供者已經把 fitting failure 的位置用 IDW 重建成完整值。

所以不能看到 NaN 就自己再補一次，也不應該直接把整片 Wafer 刪掉。我要先確認缺失到底有多少、分布在哪裡，以及兩個欄位是不是只在原本 NaN 的位置不同。


### 2.1 先看每個欄位有多少 Missing Value

先算 missing count 和 missing percentage，確認缺失是不是只集中在特定欄位，而不是整份 Measurement Data 都有問題。


In [4]:
#Missing Value Summary
missing_summary = pd.DataFrame(
    {
        "missing_count": measurement_df.isna().sum(),
        "missing_pct": (
            measurement_df.isna().mean() * 100
        ).round(2),
    }
)

missing_summary

,missing_count,missing_pct
experiment_key,0,0.0
lot_number,0,0.0
wafer_number,0,0.0
X,0,0.0
Y,0,0.0
preox_thickness,0,0.0
postox_thickness,0,0.0
postox_thickness_nan,157,2.0
stepheight,0,0.0
oxide_etch,0,0.0


**這個結果代表什麼？**

7,832 筆 records 裡，只有 `postox_thickness_nan` 有缺失：

- NaN = **157**
- Missing rate ≈ **2.0%**

其他主要欄位都是 0 missing。

所以這裡不是一般那種「很多欄位都要做 imputation」的情況，真正要處理的是這 **157 個已知的 post-etch fitting failures**。


### 2.2 `preox_thickness` 雖然沒有 NaN，但也有來源限制

`preox_thickness` 看起來 89 個位置都有值，不過資料集說明指出：89-point 版本裡只有 **15 個位置是直接量測**，其他位置是資料提供者用 IDW 重建的。

所以後面做 spatial analysis 時，要記得：

- 89 個 `preox_thickness` 不能全部當成 89 個獨立直接量測。
- `oxide_etch` 會繼承這個來源限制。
- CSV 沒有逐點標記可以告訴我哪些是原始 15 點、哪些是重建點。

這是資料集原本的 Measurement Design，不是這個專案自己另外插值。


### 2.3 建立 Post-Etch Interpolation Flag

我保留完整的 `postox_thickness` 來做後面分析，但同時新增 `postox_was_interpolated`。

只要原始 `postox_thickness_nan` 在那個位置是 NaN，就把 flag 設成 `True`。

這樣可以兼顧兩件事：數值是完整的，但我也不會忘記哪些位置不是原始 fitting 成功的 measurement。


In [5]:
#建立「是否被插值」標記
measurement_qc = measurement_df.copy()

measurement_qc["postox_was_interpolated"] = (
    measurement_qc["postox_thickness_nan"].isna()
)

measurement_qc[
    "postox_was_interpolated"
].value_counts()

postox_was_interpolated
False    7675
True      157
Name: count, dtype: int64

**這個結果代表什麼？**

Flag 分布是：

- `False`：**7,675 個位置**
- `True`：**157 個位置**

剛好和前面的 157 個 NaN 完全一致。

所以 `postox_was_interpolated=True` 真的就是在標記原始 fitting failure 的位置，沒有多標或漏標。


### 2.4 157 個缺失實際影響多少片 Wafer？

Point-level 有 157 個 NaN，不代表有 157 片 Wafer 出問題。

所以這裡再聚合到 Wafer level，看：

- 有幾片 Wafer 至少有一個 interpolation point。
- 單片最多有幾個 interpolation points。


In [6]:
#到底影響幾片 Wafer
missing_by_wafer = (
    measurement_qc
    .groupby(
        [
            "experiment_key",
            "lot_number",
            "wafer_number",
        ]
    )["postox_was_interpolated"]
    .sum()
    .reset_index(name="interpolated_points")
)

print(
    "具有至少一個插值點的 Wafer：",
    (missing_by_wafer["interpolated_points"] > 0).sum()
)

print(
    "插值點總數：",
    missing_by_wafer["interpolated_points"].sum()
)

print(
    "單片 Wafer 最多插值點數：",
    missing_by_wafer["interpolated_points"].max()
)

具有至少一個插值點的 Wafer： 20
插值點總數： 157
單片 Wafer 最多插值點數： 12


**這個結果代表什麼？**

157 個 interpolation points 分布在 **20 片 Wafer**，單片最多有 **12 個**。

所以受影響的是 20 片，不是 157 片。

而且因為資料提供者已經提供完整重建值，也沒有理由因為其中幾個位置有 fitting failure 就整片 Wafer 刪掉。比較合理的做法是保留 Wafer、保留數值，同時保留 flag。


### 2.5 這些 fitting failures 集中在哪些 Lot？

再把 interpolation point 數量按 Lot 加總，看看是不是平均分散，還是集中在某些批次。

這裡看的是 **measurement fitting failure 的分布**，不是在判斷哪個 Lot 的製程品質比較差。


In [7]:
#按 Lot 看插值問題
missing_by_lot = (
    measurement_qc
    .groupby("lot_number")[
        "postox_was_interpolated"
    ]
    .sum()
    .reset_index(name="interpolated_points")
)

missing_by_lot

,lot_number,interpolated_points
0,1,0
1,2,1
2,3,86
3,4,62
4,5,2
5,6,0
6,7,0
7,8,1
8,9,0
9,10,5


**這個結果代表什麼？**

157 個 fitting failures 很集中：

- Lot 3：**86 points**
- Lot 4：**62 points**

光這兩個 Lot 就佔了大多數。

這個結果的意義是：Lot 3、4 的 post-etch measurement fitting failure 特別多，所以後面解讀這兩個 Lot 的量測結果時，要記得它們有比較多位置是由資料提供者重建的。

但這不能直接說 Lot 3、4 的蝕刻品質比較差。品質好不好還是要在 02 用真正的 Etch 指標去看。

### 2.6 完整欄位是不是只補了原本的 NaN？

如果 `postox_thickness` 不只是補 NaN，連原本有效的 measurement 都被改過，那後面就不能簡單把它當成「完整版本」。

所以這裡只拿原本 `postox_thickness_nan` 不缺失的位置，比較兩欄是不是完全一致。


In [8]:
# 確認資料提供者的完整版本只補上原始 NaN，
# 沒有改動原本成功量測的位置。
valid_postox_mask = (
    measurement_qc["postox_thickness_nan"].notna()
)

postox_match = np.isclose(
    measurement_qc.loc[
        valid_postox_mask,
        "postox_thickness"
    ],
    measurement_qc.loc[
        valid_postox_mask,
        "postox_thickness_nan"
    ],
    atol=1e-10,
    rtol=0,
).all()

print(
    "原始非缺失值與完整版本是否一致：",
    postox_match
)


原始非缺失值與完整版本是否一致： True


**這個結果代表什麼？**

結果是 `True`。

也就是原本 fitting 成功的位置，`postox_thickness` 和 `postox_thickness_nan` 的值一致；差異只出現在原本 NaN 的位置。

所以後面可以放心用 `postox_thickness` 當完整數值，同時用 `postox_was_interpolated` 保留哪些位置是重建值。


### 2.7 拿 Excel 裡的 NaN 紀錄交叉檢查

`Lot_status.xlsx` 下半部有另外列出哪些 Lot / Wafer 發生 post-etch NaN，以及各自有幾個缺失點。

我先把那一小段整理成乾淨的 reference table，接著再和 CSV 自己算出的結果逐片比對。


In [9]:
# 讀取 Lot_status.xlsx 下半部的 NaN 紀錄

lot_nan_reference = (
    lot_status_raw
    .iloc[16:, :3]
    .copy()
)

lot_nan_reference.columns = [
    "lot_number",
    "wafer_number",
    "reference_na_count",
]

lot_nan_reference = (
    lot_nan_reference
    .dropna(how="all")
    .reset_index(drop=True)
)

for column in [
    "lot_number",
    "wafer_number",
    "reference_na_count",
]:
    lot_nan_reference[column] = pd.to_numeric(
        lot_nan_reference[column],
        errors="raise",
    ).astype(int)

lot_nan_reference

,lot_number,wafer_number,reference_na_count
0,2,3,1
1,3,3,9
2,3,4,10
3,3,5,10
4,3,6,12
5,3,7,11
6,3,8,11
7,3,9,11
8,3,10,12
9,4,1,8


**這個結果代表什麼？**

Excel reference 一共列出 **20 片 Wafer**。

這和前面直接從 CSV 算到的「20 片至少有一個 interpolation point」一致。

不過只看 Wafer 數量還不夠，下一步還要逐片確認每片的 NaN count 也完全相同。


### 2.8 逐片比對 Excel 和 CSV 的 NaN Count

把 Excel reference 和 CSV 的 missing summary 按 `lot_number + wafer_number` 一對一合併。

這樣不是只比較總數，而是可以直接確認「哪一片 Wafer 有幾個 NaN」是不是每一列都對得上。


In [10]:
# CSV 中真正有插值點的 Wafer

csv_nan_counts = (
    missing_by_wafer
    .loc[
        missing_by_wafer["interpolated_points"] > 0,
        [
            "lot_number",
            "wafer_number",
            "interpolated_points",
        ],
    ]
    .rename(
        columns={
            "interpolated_points": "csv_na_count"
        }
    )
    .reset_index(drop=True)
)


nan_crosscheck = pd.merge(
    lot_nan_reference,
    csv_nan_counts,
    on=[
        "lot_number",
        "wafer_number",
    ],
    how="outer",
    validate="one_to_one",
)

nan_crosscheck["count_match"] = (
    nan_crosscheck["reference_na_count"]
    == nan_crosscheck["csv_na_count"]
)

nan_crosscheck

,lot_number,wafer_number,reference_na_count,csv_na_count,count_match
0,2,3,1,1,True
1,3,3,9,9,True
2,3,4,10,10,True
3,3,5,10,10,True
4,3,6,12,12,True
5,3,7,11,11,True
6,3,8,11,11,True
7,3,9,11,11,True
8,3,10,12,12,True
9,4,1,8,8,True


**這個結果代表什麼？**

20 片受影響 Wafer 的 `reference_na_count` 和 `csv_na_count` 每一列都相同，`count_match` 全部都是 `True`。

所以不只是「總共 157 個」剛好一樣，連每一片 Wafer 的缺失點數都和資料提供者的 Excel 紀錄一致。


前一格已經逐片比對，這裡再做最後一個總結檢查：

- Excel 記錄幾片 Wafer。
- Excel 的 NaN 總數。
- CSV 自己算到的 NaN 總數。
- 所有逐片 count 是否全部一致。


In [11]:
print(
    "Excel 記錄 Wafer 數量：",
    len(lot_nan_reference)
)

print(
    "Excel 記錄 NaN 總數：",
    lot_nan_reference["reference_na_count"].sum()
)

print(
    "CSV 計算 NaN 總數：",
    csv_nan_counts["csv_na_count"].sum()
)

print(
    "所有 Lot / Wafer / NaN Count 是否完全一致：",
    nan_crosscheck["count_match"].all()
)

assert len(lot_nan_reference) == 20
assert lot_nan_reference["reference_na_count"].sum() == 157
assert csv_nan_counts["csv_na_count"].sum() == 157
assert nan_crosscheck["count_match"].all()

Excel 記錄 Wafer 數量： 20
Excel 記錄 NaN 總數： 157
CSV 計算 NaN 總數： 157
所有 Lot / Wafer / NaN Count 是否完全一致： True


**這個結果代表什麼？**

最後的交叉檢查全部對上：

- Excel affected wafers = **20**
- Excel NaN total = **157**
- CSV NaN total = **157**
- 每片 Wafer 的 count 全部一致 = `True`

所以這 157 個 NaN 不是這個專案讀檔或解析造成的錯誤，而是原始資料文件本來就有明確記錄的 post-etch fitting failures。


## 3. Measurement Key 和 Spatial Grid 檢查

在做任何 spatial analysis 之前，我還要確認四件事：

1. 有沒有完全重複的 rows。
2. 同一片 Wafer 的同一個 X/Y 有沒有出現兩次。
3. 88 片 Wafer 是不是都用同一組 89 個位置。
4. 這些座標有沒有跑到 200 mm Wafer 外面。

這些如果沒先確認，後面畫 Spatial Map 或逐位置比較就可能出錯。


### 3.1 Duplicate Check

一個 spatial measurement point 應該可以用：

`experiment_key + X + Y`

唯一識別。

所以我同時檢查完全重複的 row，以及 Wafer + XY key 重複。


In [12]:
full_duplicate_count = measurement_qc.duplicated().sum()

measurement_key = [
    "experiment_key",
    "X",
    "Y",
]

key_duplicate_count = (
    measurement_qc
    .duplicated(subset=measurement_key)
    .sum()
)

print("完全重複列：", full_duplicate_count)
print("Wafer + XY Key 重複：", key_duplicate_count)

完全重複列： 0
Wafer + XY Key 重複： 0


**這個結果代表什麼？**

結果是：

- 完全重複列 = **0**
- Wafer + XY key duplicate = **0**

所以每片 Wafer 的每個空間位置都只有一筆紀錄，不需要做去重。


### 3.2 88 片 Wafer 是不是都用同一個 89-point Grid？

先拿第一片 Wafer 的 X/Y set 當 reference，再逐片比較其他 87 片是不是完全相同。

這樣可以直接驗證「每片是不是同一組 spatial positions」，而不是只看每片都有 89 rows 就假設位置一定一樣。


In [13]:
first_experiment_key = measurement_qc["experiment_key"].iloc[0]

reference_positions = set(
    map(
        tuple,
        measurement_qc.loc[
            measurement_qc["experiment_key"] == first_experiment_key,
            ["X", "Y"],
        ].to_numpy(),
    )
)

spatial_grid_issues = []

for experiment_key, group in measurement_qc.groupby("experiment_key"):
    wafer_positions = set(
        map(
            tuple,
            group[["X", "Y"]].to_numpy(),
        )
    )

    if wafer_positions != reference_positions:
        spatial_grid_issues.append(experiment_key)

print(
    "Reference XY Positions：",
    len(reference_positions)
)

print(
    "Spatial Grid 不一致的 Wafer 數量：",
    len(spatial_grid_issues)
)

Reference XY Positions： 89
Spatial Grid 不一致的 Wafer 數量： 0


**這個結果代表什麼？**

Reference wafer 有 **89 個 unique XY positions**，而且 spatial grid 不一致的 Wafer 數量是 **0**。

所以現在可以比 00 更進一步確認：

> 88 片 Wafer 不只是每片有 89 rows，而是真的都使用同一組 **89 個 unique spatial positions**。

這代表後面的逐位置平均、spatial template 和 radial analysis 都可以在同一個 grid 上比較。


### 3.3 座標有沒有超出 200 mm Wafer？

200 mm Wafer 的半徑是 100 mm，也就是 100,000 µm。

所以我用 X/Y 算 radial distance，看看有沒有任何 measurement point 超出這個 nominal radius。


In [14]:
#座標是否真的在 200 mm Wafer 範圍內
measurement_qc["radius_um"] = np.sqrt(
    measurement_qc["X"] ** 2
    + measurement_qc["Y"] ** 2
)

print(
    "最大量測半徑（µm）：",
    measurement_qc["radius_um"].max()
)

print(
    "超過 100,000 µm 半徑的點數：",
    (measurement_qc["radius_um"] > 100_000).sum()
)

#代表座標空間沒有跑到 wafer 外面

最大量測半徑（µm）： 96881.37075826291
超過 100,000 µm 半徑的點數： 0


**這個結果代表什麼？**

最大的量測半徑約是 **96,881 µm = 96.88 mm**，超過 100 mm 的點數是 **0**。

所以目前沒有看到明顯落在 Wafer 外部的座標，89-point grid 的幾何位置合理。


## 4. 驗證 Derived Etch Columns 和基本數值

`oxide_etch` 和 `si_etch` 都是用其他量測欄位算出來的結果，不是另外一個感測器直接量到的值。

所以我先確認 CSV 裡的值真的符合：

- `oxide_etch = preox_thickness - postox_thickness`
- `si_etch = stepheight - postox_thickness`

這一步只是在確認資料裡的計算公式有沒有對上，不代表這兩個結果完全沒有量測不確定性。

### 4.1 先確認兩個公式有沒有對上

直接用原始組成欄位重算一次，再和 CSV 已有的 `oxide_etch`、`si_etch` 比較。


In [15]:
oxide_formula_ok = np.allclose(
    measurement_qc["oxide_etch"],
    measurement_qc["preox_thickness"]
    - measurement_qc["postox_thickness"],
    atol=1e-8,
    rtol=0,
)

si_formula_ok = np.allclose(
    measurement_qc["si_etch"],
    measurement_qc["stepheight"]
    - measurement_qc["postox_thickness"],
    atol=1e-8,
    rtol=0,
)

print(
    "oxide_etch 公式是否一致：",
    oxide_formula_ok
)

print(
    "si_etch 公式是否一致：",
    si_formula_ok
)


oxide_etch 公式是否一致： True
si_etch 公式是否一致： True


**這個結果代表什麼？**

兩個公式都是 `True`，表示 CSV 裡的 `oxide_etch` 和 `si_etch` 與組成欄位在設定的 tolerance 內一致，沒有發現計算公式錯誤。

不過資料來源限制還是要保留：

- `si_etch = stepheight - postox_thickness`，所以那 **157 個 post-etch 重建位置**的 `si_etch` 也會帶著這份不確定性。
- `oxide_etch = preox_thickness - postox_thickness`，除了 157 個 post-etch 重建位置之外，還會受到 `preox_thickness` 89-point grid 大量使用 IDW 重建的限制。
- CSV 沒有提供 pre-etch 逐點 direct/reconstructed flag，所以不能自己精確標出所有 pre-etch 重建位置。

另外，以後如果 `oxide_etch` 或 `si_etch` 當 target，就不能再把能直接算出 target 的 metrology components 當 predictor，不然會造成 target leakage。


### 4.2 做最基本的數值合理性檢查

這裡只排除很明顯不合理的情況，例如負厚度、負 stepheight 或負 etch depth。

這不是製造規格檢查；全部通過也不代表 Wafer 就是合格品。


In [16]:
#基本數值合理性
#有沒有非常明顯不可能的值？
basic_range_checks = pd.DataFrame(
    {
        "檢查項目": [
            "preox_thickness > 0",
            "postox_thickness > 0",
            "stepheight > 0",
            "oxide_etch >= 0",
            "si_etch >= 0",
        ],
        "是否全部通過": [
            (measurement_qc["preox_thickness"] > 0).all(),
            (measurement_qc["postox_thickness"] > 0).all(),
            (measurement_qc["stepheight"] > 0).all(),
            (measurement_qc["oxide_etch"] >= 0).all(),
            (measurement_qc["si_etch"] >= 0).all(),
        ],
    }
)

basic_range_checks

#只是證明目前沒有最基本的負厚度、負 etch depth 這種明顯錯值。

,檢查項目,是否全部通過
0,preox_thickness > 0,True
1,postox_thickness > 0,True
2,stepheight > 0,True
3,oxide_etch >= 0,True
4,si_etch >= 0,True


**這個結果代表什麼？**

五項基本檢查全部都是 `True`：

- preox thickness > 0
- postox thickness > 0
- stepheight > 0
- oxide etch >= 0
- silicon etch >= 0

所以目前沒有看到明顯不可能的數值。

但這只能排除最基本的資料錯誤，不能拿來判斷製程是否異常或品質是否合格。


### 4.3 先記錄 Measurement 的實際數值範圍

把主要欄位的 min、mean、std、max 存成描述統計，讓 02 做 Wafer Quality Analysis 時有一個整體尺度可以對照。


In [17]:
#看 Measurement 數值範圍
measurement_numeric_summary = (
    measurement_qc[
        [
            "preox_thickness",
            "postox_thickness",
            "stepheight",
            "oxide_etch",
            "si_etch",
        ]
    ]
    .describe()
    .T
    [
        [
            "min",
            "mean",
            "std",
            "max",
        ]
    ]
)

measurement_numeric_summary

,min,mean,std,max
preox_thickness,0.9916,1.001726,0.003312,1.024100
postox_thickness,0.1686,0.343578,0.076287,0.569800
stepheight,28.9455,44.011698,3.615865,53.207300
oxide_etch,0.4248,0.658148,0.077312,0.838006
si_etch,28.6530,43.668120,3.609935,52.780900


**這個結果代表什麼？**

這張表只是告訴我目前資料實際落在哪個範圍，不是規格上下限。

例如 `si_etch` 在 point level 大約介於 **28.6530 到 52.7809**，平均約 **43.6681**。

看到接近 min 或 max 的值時，現在還不能直接叫異常；02 會把資料拉回 Wafer level，再從 Lot、sequence 和 spatial pattern 判斷這些極端值的意義。


## 5. 整理 Lot Metadata

`Lot_status.xlsx` 的內容本身可以用，但文字格式不完全一致，例如：

- `2th July 2024` 這類日期寫法
- Conditioning 前後多餘空白
- Measurement 字串的空白數量不同

這些不是實驗內容錯誤，但會影響 merge 和 groupby，所以我把格式整理乾淨，同時保留原始文字方便追查。


### 5.1 先整理基本欄位

先抓 Lot 1–10：

- 建立數字型 `lot_number`
- 去掉字串前後空白
- 把 wafer count 轉成 integer

這一格先做最基本的格式整理。


In [18]:
#重新整理 Lot Summary
lot_summary_clean = lot_status_raw.iloc[
    1:11
].copy()

lot_summary_clean.columns = [
    "lot",
    "date_raw",
    "wafers",
    "measurements",
    "conditioning",
]

lot_summary_clean["lot_number"] = (
    lot_summary_clean["lot"]
    .str.extract(r"(\d+)")
    .astype(int)
)

lot_summary_clean["measurements"] = (
    lot_summary_clean["measurements"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

lot_summary_clean["conditioning"] = (
    lot_summary_clean["conditioning"]
    .astype(str)
    .str.strip()
)

# Wafer 數量轉成整數
lot_summary_clean["wafers"] = pd.to_numeric(
    lot_summary_clean["wafers"],
    errors="raise",
).astype(int)

lot_summary_clean

,lot,date_raw,wafers,measurements,conditioning,lot_number
1,Lot 1,2th July 2024,10,89 & 9 points,3C,1
2,Lot 2,5th July 2024,10,89 & 9 points,1C,2
3,Lot 3,9th July 2024,10,89 & 9 points,9C,3
4,Lot 4,11th July 2024,10,89 & 9 points,3C - Si,4
5,Lot 5,19th July 2024,10,89 & 9 points,1C - Si,5
6,Lot 6,1st Aug 2024,10,89 & 9 points,9C - Si,6
7,Lot 7,5th Aug 2024,6,89 & 9 points,3C - SiO2,7
8,Lot 8,7th Aug 2024,10,89 points,3C - SiO2,8
9,Lot 9,21st Aug 2024,10,89 & 9 points,3C,9
10,Lot 10,22nd Aug 2024,10,89 points,3C - SiO2,10


**這個結果代表什麼？**

10 個 Lot 都成功整理成一致欄位，Lot number 和 wafer count 也已經是後面方便運算的格式。

原始的 `date_raw` 還保留，所以格式清理後仍然可以回頭追到原始文字。


### 5.2 把日期轉成真正的日期格式

原始日期有 `1st`、`2th`、`11th` 這種序數字尾，先把字尾移掉，再用 `pd.to_datetime(..., dayfirst=True)` 轉成標準日期。


In [19]:
# 移除日期中的英文序數字尾，例如：
# 2th July 2024 -> 2 July 2024
# 1st Aug 2024  -> 1 Aug 2024

def remove_ordinal_suffix(value):
    return re.sub(
        r"(\d+)(st|nd|rd|th)",
        r"\1",
        str(value).strip(),
    )


lot_summary_clean["date_clean"] = (
    lot_summary_clean["date_raw"]
    .apply(remove_ordinal_suffix)
)


# 原始日期同時包含 July、Aug 等不同月份寫法，
# 因此使用 format="mixed" 讓 pandas 逐筆判斷日期格式。
lot_summary_clean["date"] = pd.to_datetime(
    lot_summary_clean["date_clean"],
    format="mixed",
    dayfirst=True,
)


lot_summary_clean[
    [
        "lot_number",
        "date_raw",
        "date_clean",
        "date",
        "wafers",
        "measurements",
        "conditioning",
    ]
]

,lot_number,date_raw,date_clean,date,wafers,measurements,conditioning
1,1,2th July 2024,2 July 2024,2024-07-02,10,89 & 9 points,3C
2,2,5th July 2024,5 July 2024,2024-07-05,10,89 & 9 points,1C
3,3,9th July 2024,9 July 2024,2024-07-09,10,89 & 9 points,9C
4,4,11th July 2024,11 July 2024,2024-07-11,10,89 & 9 points,3C - Si
5,5,19th July 2024,19 July 2024,2024-07-19,10,89 & 9 points,1C - Si
6,6,1st Aug 2024,1 Aug 2024,2024-08-01,10,89 & 9 points,9C - Si
7,7,5th Aug 2024,5 Aug 2024,2024-08-05,6,89 & 9 points,3C - SiO2
8,8,7th Aug 2024,7 Aug 2024,2024-08-07,10,89 points,3C - SiO2
9,9,21st Aug 2024,21 Aug 2024,2024-08-21,10,89 & 9 points,3C
10,10,22nd Aug 2024,22 Aug 2024,2024-08-22,10,89 points,3C - SiO2


**這個結果代表什麼？**

Lot 1–10 都成功轉成 **2024-07-02 到 2024-08-22** 的標準日期。

`date_raw` 和清理過的 `date_clean` 都還保留，所以現在既有可以直接比較和 merge 的日期，也沒有把原始資料來源丟掉。


### 5.3 整理後的 Lot Summary 還完整嗎？

格式清理完後，再確認：

- 還是 10 個 Lots
- Wafer 總數還是 96
- 關鍵 metadata 沒有因清理過程產生 missing


In [20]:
#Lot Status 基本檢查
print(
    "Lot 數量：",
    lot_summary_clean["lot_number"].nunique()
)

print(
    "Wafer 總數：",
    int(lot_summary_clean["wafers"].sum())
)

print(
    "缺失值總數：",
    lot_summary_clean[
        [
            "lot_number",
            "date",
            "wafers",
            "measurements",
            "conditioning",
        ]
    ]
    .isna()
    .sum()
    .sum()
)

Lot 數量： 10
Wafer 總數： 96
缺失值總數： 0


**這個結果代表什麼？**

結果和 00 完全一致：

- Lot count = **10**
- Wafer total = **96**
- 關鍵 metadata missing = **0**

所以 Lot metadata 的格式整理沒有把 Lot 刪掉、Wafer 數改掉或產生新的缺失。


### 5.4 把 Conditioning 拆成 Count 和 Surface

原始 Conditioning 像：

- `3C`
- `3C - Si`
- `3C - SiO2`

一個字串同時混了 conditioning 次數和使用的 surface。

所以我把它拆成：

- `conditioning_count`
- `conditioning_surface`

後面要分析時，才可以分開看「做幾次」和「在哪種 surface 上做」。


In [21]:
# 先建立完整格式規則，避免未知字串被默認分類成 Chuck。
conditioning_text = (
    lot_summary_clean["conditioning"]
    .astype(str)
    .str.strip()
)

conditioning_pattern = (
    r"^(?P<count>\d+)C"
    r"(?:\s*-\s*(?P<surface>SiO2|Si))?$"
)

conditioning_parts = (
    conditioning_text
    .str.extract(conditioning_pattern)
)

# 任何無法解析的字串都應直接停止，而不是 silent fallback。
conditioning_parse_ok = (
    conditioning_parts["count"].notna().all()
)

assert conditioning_parse_ok, (
    "Unrecognized conditioning format found in Lot_status.xlsx."
)

lot_summary_clean["conditioning_count"] = (
    conditioning_parts["count"]
    .astype(int)
)

lot_summary_clean["conditioning_surface"] = (
    conditioning_parts["surface"]
    .map(
        {
            "Si": "Si wafer",
            "SiO2": "SiO2 wafer",
        }
    )
    .fillna("Chuck")
)

assert (
    lot_summary_clean["conditioning_surface"]
    .notna()
    .all()
), "Conditioning surface parsing failed."

lot_summary_clean[
    [
        "lot_number",
        "conditioning",
        "conditioning_count",
        "conditioning_surface",
    ]
]


,lot_number,conditioning,conditioning_count,conditioning_surface
1,1,3C,3,Chuck
2,2,1C,1,Chuck
3,3,9C,9,Chuck
4,4,3C - Si,3,Si wafer
5,5,1C - Si,1,Si wafer
6,6,9C - Si,9,Si wafer
7,7,3C - SiO2,3,SiO2 wafer
8,8,3C - SiO2,3,SiO2 wafer
9,9,3C,3,Chuck
10,10,3C - SiO2,3,SiO2 wafer


**這個結果代表什麼？**

Conditioning 成功拆成：

- Count：**1、3、9**
- Surface：**Chuck、Si wafer、SiO2 wafer**

另外 parser 不是遇到不認得的字串就默認成 Chuck，而是只有符合預期格式才通過。

這樣如果未來 Excel 出現新的寫法或錯字，程式會直接停下來，不會在沒有報錯的情況下把資料分錯類。

這裡只做 metadata 整理，還沒有在檢驗 Conditioning 對品質或製程的效果。


## 6. Process Data 品質檢查

00 已經確認 96 片 Wafer 共同都有 **31 個 Process Features**。

所以 01 不再重新探索 31/44 欄版本，而是針對這 31 個共同欄位檢查：

- NaN / Inf
- timestamp 是否遞增
- sampling interval
- large gap
- constant features

前面那 13 個只出現在 2024-07-02、而且全部固定的欄位不再放進共同分析。


### 6.1 先重新確認共同 Features 還是 31 個

這裡再求一次 96 片的 Feature 交集，不是重做 00 的 EDA，而是保險起見。

如果 Raw Process Data 換了版本，01 應該在一開始就發現「96 wafers / 31 common features」已經不成立，而不是用舊假設繼續往下跑。


In [22]:
#重新取得共同 31 欄
def clean_feature_name(value):
    if isinstance(value, bytes):
        return value.decode("utf-8")
    return str(value)


all_feature_sets = []

with Dataset(process_path, "r") as process_ds:
    process_groups = list(
        process_ds.groups.keys()
    )

    for group in process_ds.groups.values():
        features = {
            clean_feature_name(value)
            for value in group["feature"][:]
        }

        all_feature_sets.append(features)


common_process_features = sorted(
    set.intersection(*all_feature_sets)
)

print(
    "Process Wafer 數量：",
    len(process_groups)
)

print(
    "所有 Wafer 共同 Features：",
    len(common_process_features)
)

Process Wafer 數量： 96
所有 Wafer 共同 Features： 31


**這個結果代表什麼？**

Process Data 目前仍然是：

- **96 個 Wafer groups**
- **31 個共同 Process Features**

所以 00 的欄位結構在這次執行中仍然成立，可以接著做 Process QC。


### 6.2 一次掃描 Process 數值和時間軸

對每片 Wafer，我都做同樣的檢查：

1. 只取 31 個共同 Features。
2. 用 dictionary 解回真實 Process values。
3. 計算 NaN / Inf。
4. 算相鄰 timestamp 的差。
5. 記錄 median interval、max interval、non-positive interval 和 >1 秒 gap。
6. 同時記每個 Feature 的 global min/max，後面檢查 constant feature。

這裡的 >1 秒只是 QC threshold，還不代表那片 Wafer 就有製程問題。


In [23]:
process_qc_records = []
process_feature_range_records = []

with Dataset(dictionary_path, "r") as dictionary_ds:
    decoder = dictionary_ds["data"][:]

with Dataset(process_path, "r") as process_ds:

    for group_name, group in process_ds.groups.items():

        features = [
            clean_feature_name(value)
            for value in group["feature"][:]
        ]

        feature_index = {
            feature: i
            for i, feature in enumerate(features)
        }

        common_indices = [
            feature_index[feature]
            for feature in common_process_features
        ]

        encoded = group["data"][:, common_indices]
        decoded = decoder[encoded]

        times = np.asarray(
            group["times"][:],
            dtype=float,
        )

        time_diff = np.diff(times)

        process_qc_records.append(
            {
                "group_name": group_name,
                "n_timepoints": len(times),
                "nan_count": int(np.isnan(decoded).sum()),
                "inf_count": int(np.isinf(decoded).sum()),
                "median_dt": float(np.median(time_diff)),
                "max_dt": float(np.max(time_diff)),
                "nonpositive_dt": int((time_diff <= 0).sum()),
                "large_gap_count": int(
                    (time_diff > LARGE_GAP_THRESHOLD_SECONDS).sum()
                ),
            }
        )

        for j, feature in enumerate(common_process_features):
            values = decoded[:, j]

            process_feature_range_records.append(
                {
                    "group_name": group_name,
                    "feature": feature,
                    "min_value": values.min(),
                    "max_value": values.max(),
                }
            )

process_qc_df = pd.DataFrame(process_qc_records)
process_feature_ranges = pd.DataFrame(process_feature_range_records)

process_qc_df.head()

,group_name,n_timepoints,nan_count,inf_count,median_dt,max_dt,nonpositive_dt,large_gap_count
0,Day_2024_07_02_Wafer_01,3245,0,0,0.2,42.88,0,1
1,Day_2024_07_02_Wafer_02,3298,0,0,0.2,42.77,0,1
2,Day_2024_07_02_Wafer_03,3247,0,0,0.2,42.44,0,1
3,Day_2024_07_02_Wafer_04,3247,0,0,0.2,42.50,0,1
4,Day_2024_07_02_Wafer_05,3298,0,0,0.2,42.79,0,1


**這個結果代表什麼？**

前幾片 Wafer 都可以先看到：

- `median_dt = 0.2 s`
- NaN = **0**
- Inf = **0**
- timestamp 是往前遞增的
- 但部分 Wafer 的 `max_dt` 明顯很大

所以 Process signal 本身看起來沒有一般 missing/Inf 問題，但時間軸可能有 gap。

接下來要把 timestamp metadata、數值缺失、constant feature 和 gap 分開看，不能全部混成同一種 Data Quality 問題。


### 6.3 Timestamp Metadata 到底能不能當真實實驗時間？

這一步很重要，因為後面的 time-series analysis 都會用到 `times`。

我掃描全部 96 個 Process groups，逐片記錄：

- `times.units`
- `calendar`
- Group Name 裡的實驗日期
- 第一個和最後一個 numeric time value
- 按照每個 Group 自己的 NetCDF metadata 解碼後得到的日期
- metadata 能不能成功解碼

我要分開確認兩件事：

1. metadata 本身能不能正常解碼。
2. 解碼成功後，日期是否真的對得上 Group Name 裡的 2024 實驗日期。

這樣可以避免 metadata 其實解碼失敗，卻因為產生 `NaT` 而誤以為只是「日期不一致」。


In [24]:
# 檢查全部 96 個 Process groups 的 Timestamp Metadata
# 這裡直接使用每個 time variable 自己的 units / calendar 做 NetCDF 解碼，
# 不再只針對某一個特定 units 字串手動換算。
time_metadata_records = []

group_name_pattern = re.compile(
    r"^Day_(\d{4}_\d{2}_\d{2})_Wafer_(\d+)$"
)

with Dataset(process_path, "r") as process_ds:
    for group_name in process_groups:
        group = process_ds.groups[group_name]
        time_variable = group["times"]

        match = group_name_pattern.match(group_name)
        if match is None:
            raise ValueError(
                f"Unexpected Process group name: {group_name}"
            )

        experiment_date = pd.to_datetime(
            match.group(1).replace("_", "-")
        ).normalize()

        time_units = getattr(time_variable, "units", None)
        calendar = getattr(time_variable, "calendar", "standard")

        if not isinstance(time_units, str) or not time_units.strip():
            raise ValueError(
                f"{group_name} 的 times.units 缺失或不是有效字串"
            )

        if not isinstance(calendar, str) or not calendar.strip():
            calendar = "standard"

        times = np.asarray(time_variable[:], dtype=float)

        if len(times) == 0:
            raise ValueError(f"{group_name} 的 times 為空")
        if not np.isfinite(times).all():
            raise ValueError(f"{group_name} 的 times 含 NaN / Inf")

        first_time = float(times[0])
        last_time = float(times[-1])

        try:
            decoded_first = num2date(
                first_time,
                units=time_units,
                calendar=calendar,
                only_use_cftime_datetimes=False,
            )

            # 只需要比較 calendar date，因此轉成 YYYY-MM-DD 再交給 pandas。
            metadata_implied_date = pd.Timestamp(
                decoded_first.strftime("%Y-%m-%d")
            ).normalize()

            decode_ok = True
            decode_error = None

        except Exception as exc:
            metadata_implied_date = pd.NaT
            decode_ok = False
            decode_error = repr(exc)

        time_metadata_records.append(
            {
                "group_name": group_name,
                "experiment_date": experiment_date,
                "time_units": time_units,
                "calendar": calendar,
                "first_time": first_time,
                "last_time": last_time,
                "metadata_decode_ok": decode_ok,
                "metadata_decode_error": decode_error,
                "metadata_implied_date": metadata_implied_date,
            }
        )

time_metadata_df = pd.DataFrame(time_metadata_records)

time_metadata_df["date_matches_metadata"] = (
    time_metadata_df["experiment_date"]
    == time_metadata_df["metadata_implied_date"]
)

# 先確認 metadata decoding 本身真的成功，不能讓 NaT 比較造成假通過。
if not time_metadata_df["metadata_decode_ok"].all():
    failed = time_metadata_df.loc[
        ~time_metadata_df["metadata_decode_ok"],
        ["group_name", "time_units", "calendar", "metadata_decode_error"],
    ]
    display(failed)
    raise ValueError(
        "至少一個 Process group 的 timestamp metadata 無法依 NetCDF 規則解碼"
    )

if time_metadata_df["metadata_implied_date"].isna().any():
    raise ValueError(
        "Timestamp metadata decoding 後仍出現 NaT，不能繼續做日期一致性結論"
    )

timestamp_metadata_summary = pd.DataFrame(
    {
        "metric": [
            "Process groups",
            "Unique time units",
            "Unique calendars",
            "Metadata decode success",
            "Minimum first_time",
            "Maximum first_time",
            "Minimum last_time",
            "Maximum last_time",
            "Experiment date matches metadata-implied date",
        ],
        "value": [
            len(time_metadata_df),
            time_metadata_df["time_units"].nunique(dropna=False),
            time_metadata_df["calendar"].nunique(dropna=False),
            int(time_metadata_df["metadata_decode_ok"].sum()),
            time_metadata_df["first_time"].min(),
            time_metadata_df["first_time"].max(),
            time_metadata_df["last_time"].min(),
            time_metadata_df["last_time"].max(),
            int(time_metadata_df["date_matches_metadata"].sum()),
        ],
    }
)

assert len(time_metadata_df) == 96
assert time_metadata_df["time_units"].nunique(dropna=False) == 1
assert time_metadata_df["calendar"].nunique(dropna=False) == 1
assert time_metadata_df["metadata_decode_ok"].all()
assert not time_metadata_df["date_matches_metadata"].any(), (
    "Process times unexpectedly align with experiment calendar dates."
)

print("Unique times.units：")
display(time_metadata_df[["time_units"]].drop_duplicates().reset_index(drop=True))

print("Unique calendar：")
display(time_metadata_df[["calendar"]].drop_duplicates().reset_index(drop=True))

print("Timestamp metadata summary：")
display(timestamp_metadata_summary)


Unique times.units：


,time_units
0,seconds since 1970-01-01 00:00:00 UTC


Unique calendar：


,calendar
0,standard


Timestamp metadata summary：


,metric,value
0,Process groups,96.00
1,Unique time units,1.00
2,Unique calendars,1.00
3,Metadata decode success,96.00
4,Minimum first_time,0.00
5,Maximum first_time,142.83
6,Minimum last_time,690.67
7,Maximum last_time,829.95
8,Experiment date matches metadata-implied date,0.00


**這個結果代表什麼？**

這裡得到一個很重要的限制。

96 個 Group 的：

- `times.units` 全部一致
- `calendar` 全部一致
- metadata decoding **96 / 96 都成功**

所以 metadata 本身不是壞掉或讀不到。

但解碼後的 calendar date 和 Group Name 裡的 2024 實驗日期：

- Matches = **0 / 96**

也就是說，NetCDF 的 `times` 雖然可以當數字時間軸使用，但**不能當成可靠的 absolute experiment timestamp**。

後面可以用它來算：

- sampling interval
- duration
- active-window segmentation
- slope / time-series features

真正的實驗日期則要用 Group Name 或 Lot metadata。

這個結果很重要，因為如果把 `times` 誤當真實 Unix timestamp，後面的日期分析會整個錯掉。


### 6.4 Process signal 有沒有 NaN、Inf 或時間倒退？

把 96 片 Wafer 的 QC summary 加總，直接確認共同 31 個 Features 有沒有數值缺失，以及相鄰 timestamp 有沒有 <= 0 的情況。


In [25]:
#Process Missing / Infinite 結果
print(
    "Process NaN 總數：",
    process_qc_df["nan_count"].sum()
)

print(
    "Process Infinite 總數：",
    process_qc_df["inf_count"].sum()
)

print(
    "時間非遞增紀錄總數：",
    process_qc_df["nonpositive_dt"].sum()
)

Process NaN 總數： 0
Process Infinite 總數： 0
時間非遞增紀錄總數： 0


**這個結果代表什麼？**

總結果是：

- Process NaN = **0**
- Process Inf = **0**
- Non-positive time differences = **0**

所以 31 個共同 Process Features 不需要做一般的 missing-value imputation。

而且每片 Wafer 的 numeric time axis 都是往前走的，沒有時間倒退或重複 timestamp 的問題。


### 6.5 找出完全沒有變化的 Process Features

即使沒有 NaN，有些 signal 如果從頭到尾、跨所有 Wafer 都完全不變，也沒有分析價值。

所以對 31 個共同 Features 計算全資料的 global min 和 max。只有 `global_min == global_max` 才叫 constant feature。


In [26]:
#找共同 31 欄裡的 Constant Features
global_feature_ranges = (
    process_feature_ranges
    .groupby("feature")
    .agg(
        global_min=("min_value", "min"),
        global_max=("max_value", "max"),
    )
)

global_feature_ranges["is_constant"] = (
    global_feature_ranges["global_min"]
    == global_feature_ranges["global_max"]
)

constant_process_features = (
    global_feature_ranges[
        global_feature_ranges["is_constant"]
    ]
)

constant_process_features

,global_min,global_max,is_constant
feature,,,
Stat3_Etch_MV_Gas3Flow,0.0,0.0,True
Stat3_Etch_MV_Gas8Flow,0.0,0.0,True
Stat3_Etch_MV_SourceRF2LoadPower,0.0,0.0,True
Stat3_Etch_MV_SourceRF2ReflectedPower,0.0,0.0,True


**這個結果代表什麼？**

31 個共同 Features 裡有 **4 個欄位全域固定為 0**：

- `Stat3_Etch_MV_Gas3Flow`
- `Stat3_Etch_MV_Gas8Flow`
- `Stat3_Etch_MV_SourceRF2LoadPower`
- `Stat3_Etch_MV_SourceRF2ReflectedPower`

因為它們完全沒有時間變化，也沒有 Wafer-to-Wafer 差異，所以不可能幫助解釋製程變化。

後續分析會排除這 4 欄，但 Raw Data 不會刪除。


### 6.6 Sampling Interval 和 Gap 整體長什麼樣？

接著看 96 片 Wafer 的：

- timepoint count
- median interval
- maximum interval
- large gap count

我主要想確認 0.2 秒的約 5 Hz 取樣是不是每片都成立，以及 large gap 是少數特例還是很常見。


In [27]:
#檢查 5 Hz
process_qc_df[
    [
        "n_timepoints",
        "median_dt",
        "max_dt",
        "large_gap_count",
    ]
].describe()

,n_timepoints,median_dt,max_dt,large_gap_count
count,96.000000,96.0,96.000000,96.000000
mean,3262.177083,0.2,30.277604,0.718750
std,67.733533,0.0,18.920199,0.451969
min,3193.000000,0.2,0.200000,0.000000
25%,3246.000000,0.2,0.210000,0.000000
50%,3247.000000,0.2,41.635000,1.000000
75%,3297.000000,0.2,42.440000,1.000000
max,3835.000000,0.2,45.200000,1.000000


**這個結果代表什麼？**

96 片 Wafer 的 `median_dt` 全部都是 **0.2 秒**，所以一般情況下確實符合約 **5 Hz** 的取樣節奏。

但 `max_dt` 最大到約 **45.2 秒**，而 `large_gap_count` 每片是 0 或 1。

也就是一般 sampling interval 很穩定，但不少 Wafer 會出現一個很大的時間跳躍。下一步要看這個 gap 是發生在製程中段，還是固定發生在尾端。


### 6.7 有多少片 Wafer 真的出現 >1 秒 Gap？

直接統計至少有一個 large gap 的 Wafer 數量，看看這件事有多普遍。


In [28]:
#到底多少 Wafer 有時間跳躍？
wafers_with_large_gap = (
    process_qc_df["large_gap_count"] > 0
).sum()

print(
    "具有 > 1 秒時間跳躍的 Wafer：",
    wafers_with_large_gap
)

print(
    "總 Process Wafer：",
    len(process_qc_df)
)

具有 > 1 秒時間跳躍的 Wafer： 69
總 Process Wafer： 96


**這個結果代表什麼？**

96 片 Process Wafers 裡有 **69 片**至少出現一個 >1 秒 gap。

這不是只有一兩片的偶發資料點，而是相當常見的結構。

所以不能簡單把所有有 gap 的 Wafer 都當成壞資料；先找 gap 的位置比較重要。


### 6.8 Large Gap 都發生在哪裡？

把所有 >1 秒 gap 的 index 找出來，看看它是不是都出現在：

> 倒數第二個 sample → 最後一個 sample

如果全部在尾端，它和「製程中間少了一段 samples」的意義完全不同。


In [29]:
gap_position_records = []

with Dataset(process_path, "r") as process_ds:

    for group_name, group in process_ds.groups.items():

        times = np.asarray(
            group["times"][:],
            dtype=float,
        )

        time_diff = np.diff(times)

        large_gap_indices = np.where(
            time_diff > LARGE_GAP_THRESHOLD_SECONDS
        )[0]

        for gap_index in large_gap_indices:
            gap_position_records.append(
                {
                    "group_name": group_name,
                    "gap_seconds": time_diff[gap_index],
                    "gap_before_final_sample":
                        gap_index == len(time_diff) - 1,
                }
            )

gap_position_df = pd.DataFrame(gap_position_records)

print(
    "Large Gap 總數：",
    len(gap_position_df)
)

print(
    "是否全部發生在最後一筆資料之前：",
    gap_position_df["gap_before_final_sample"].all()
)

Large Gap 總數： 69
是否全部發生在最後一筆資料之前： True


**這個結果代表什麼？**

一共有 **69 個 large gaps**，而且「是否全部發生在最後一筆之前」是 `True`。

也就是 69 個 gap 全部都是最後一次 transition，gap 後面只剩最後一筆 sample。

所以目前最合理的處理不是把它當成製程中段缺資料，也不是直接把最後一筆刪掉，而是：

- 保留原始資料
- 保留 gap QC flag
- 後面算 duration、slope 或 time-weighted feature 時，不讓這個 terminal gap 被當成正常製程時間

這個結果描述的是資料時間結構，不代表 69 片 Wafer 都是異常製程。


### 6.9 建立後面真正會用的 Process Feature List

31 個共同 Features 裡，我只先排除前面確認「完全 constant」的 4 個。

Near-zero variance 這時還不急著刪，因為某些 plasma signal 可能只在特定 process phase 才有變化。如果現在用整段時間序列的 variance 太早過濾，可能會把有用的 phase information 刪掉。


In [30]:
#建立真正可用的 Process Feature List
constant_feature_names = (
    constant_process_features
    .index
    .tolist()
)

usable_process_features = [
    feature
    for feature in common_process_features
    if feature not in constant_feature_names
]

print(
    "共同 Process Features：",
    len(common_process_features)
)

print(
    "Constant Features：",
    len(constant_feature_names)
)

print(
    "後續可分析 Process Features：",
    len(usable_process_features)
)

共同 Process Features： 31
Constant Features： 4
後續可分析 Process Features： 27


**這個結果代表什麼？**

最後得到：

- Common Process Features = **31**
- Constant Features = **4**
- Usable Process Features = **27**

所以後面的 Process EDA、SPC 和 Feature Engineering 都從這 **27 個有實際變化的共同 signals** 開始。


再把被排除的 4 個 constant features 明確列出來。

這樣後面的 Notebook 不用猜「哪四個被刪掉」，也能確認只有這四個因為全域固定而被排除。


In [31]:
print("後續排除的 Constant Process Features：\n")

for feature in constant_feature_names:
    print("-", feature)

後續排除的 Constant Process Features：

- Stat3_Etch_MV_Gas3Flow
- Stat3_Etch_MV_Gas8Flow
- Stat3_Etch_MV_SourceRF2LoadPower
- Stat3_Etch_MV_SourceRF2ReflectedPower


**這個結果代表什麼？**

列出的 4 個 Features 和前面的 constant-feature table 完全一致，沒有偷偷多刪其他 signal。

Near-zero variance 會留到真正產生 Wafer-level aggregate features 後再判斷，這樣比較不會過早丟掉 phase-specific 資訊。


## 7. 建立後面可以直接接著用的 Interim Data

到這裡已經完成：

- interpolation flag
- spatial radius
- Lot metadata 整理
- usable Process Feature list
- Process QC summary

接著只是把這些已經驗證過的結果整合和輸出，不在這一節再開新的 EDA。


### 7.1 合併 Lot Metadata 前，再驗證一次日期

只用 `lot_number` merge 雖然方便，但如果 Lot 編號或日期對錯，也有可能「merge 成功卻接錯資料」。

所以我先：

- 從 `experiment_key` 抽出日期
- 按 `lot_number` 對到 Lot Status 的標準日期
- 確認兩邊逐列一致

日期沒問題後，再建立：

- `x_mm`
- `y_mm`
- `radius_mm`

最後用 `validate="many_to_one"` 合併 Lot metadata，避免 Lot table 有重複 key 時把 Measurement rows 意外乘開。


In [32]:
# 先做 experiment_key date ↔ Lot date 的跨檔案一致性檢查
experiment_date_from_key = pd.to_datetime(
    measurement_qc["experiment_key"]
    .str.extract(
        r"^(\d{4}-\d{2}-\d{2})_",
        expand=False,
    ),
    errors="raise",
)

lot_date_lookup = (
    lot_summary_clean
    .set_index("lot_number")["date"]
)

expected_lot_date = (
    measurement_qc["lot_number"]
    .map(lot_date_lookup)
)

date_match = (
    experiment_date_from_key
    == expected_lot_date
)

print(
    "Experiment date 與 Lot date 不一致的 rows：",
    int((~date_match).sum()),
)

assert date_match.all(), (
    "experiment_key date does not match Lot_status date."
)

# 建立 mm 單位的 spatial columns
measurement_qc["x_mm"] = (
    measurement_qc["X"] / 1000
)

measurement_qc["y_mm"] = (
    measurement_qc["Y"] / 1000
)

measurement_qc["radius_mm"] = (
    measurement_qc["radius_um"] / 1000
)

# 合併標準化 Lot metadata
measurement_interim = pd.merge(
    measurement_qc,
    lot_summary_clean[
        [
            "lot_number",
            "date",
            "conditioning",
            "conditioning_count",
            "conditioning_surface",
        ]
    ],
    on="lot_number",
    how="left",
    validate="many_to_one",
)

print(
    "合併後資料維度：",
    measurement_interim.shape
)

print(
    "合併後 Lot Metadata Missing：",
    measurement_interim[
        [
            "date",
            "conditioning",
            "conditioning_count",
            "conditioning_surface",
        ]
    ]
    .isna()
    .sum()
    .sum()
)


Experiment date 與 Lot date 不一致的 rows： 0
合併後資料維度： (7832, 20)
合併後 Lot Metadata Missing： 0


**這個結果代表什麼？**

合併前先比對日期，結果是 **0 筆不一致**，表示 `experiment_key` 裡的日期和 `Lot_status.xlsx` 的 Lot 日期都能對上。

合併完成後：

- 資料維度仍是 **7,832 × 20**
- Lot metadata missing = **0**
- row 數沒有增加，也沒有減少

這個結果的重點是：不是只有 `lot_number` 剛好可以 merge，而是連 Wafer key 裡的日期也一致，所以這次 metadata 合併沒有把資料接到錯的 Lot。


### 7.2 把所有 Data Quality 發現整理成處理規則

到這裡問題已經找完了，我把每一類問題和處理方法整理成 Decision Table。

我的原則不是看到問題就刪資料，而是分成：

- 保留
- 建立 flag
- 標準化
- 只在特定分析中排除

Raw Data 一律不修改。


In [33]:
data_quality_decisions = pd.DataFrame(
    [
        {
            "項目": "postox_thickness_nan",
            "發現": "157 個原始 fitting-failure NaN，分布在 20 片 Wafer",
            "處理": "保留原始欄位並建立 postox_was_interpolated Flag",
        },
        {
            "項目": "postox_thickness",
            "發現": "原作者已對 fitting-failure positions 使用 IDW reconstruction",
            "處理": "使用完整欄位，但保留 interpolation provenance",
        },
        {
            "項目": "preox_thickness",
            "發現": "89-point grid 中只有 15 個直接量測點，其餘由資料提供者 IDW 重建",
            "處理": "保留；Spatial / oxide_etch 解讀必須註明 provenance 限制",
        },
        {
            "項目": "Duplicate Measurement",
            "發現": "0 筆 full-row duplicate；0 筆 Wafer+XY key duplicate",
            "處理": "不需去重",
        },
        {
            "項目": "Spatial Grid",
            "發現": "88 片 Wafer 均使用相同 89 個 unique XY positions",
            "處理": "可在共同 grid 上進行 Wafer Spatial Analysis",
        },
        {
            "項目": "Derived Etch Columns",
            "發現": "oxide_etch 與 si_etch 公式驗證一致，並繼承組成量測欄位的 reconstruction uncertainty",
            "處理": "可作為 Outcome；保留 provenance，且避免把可直接計算 target 的 metrology components 當 predictors",
        },
        {
            "項目": "Lot Metadata",
            "發現": "日期、空格與 conditioning 格式已標準化，experiment_key date 與 Lot date 已交叉驗證",
            "處理": "使用標準化欄位；未知 conditioning 格式直接停止",
        },
        {
            "項目": "Process Missing / Inf",
            "發現": "31 個共同 Process Features 中 NaN = 0、Inf = 0",
            "處理": "不需補值",
        },
        {
            "項目": "Constant Process Features",
            "發現": "31 個共同欄位中有 4 個全域 constant",
            "處理": "後續分析排除，Raw Data 保留",
        },
        {
            "項目": "Process Terminal Gap",
            "發現": "69 / 96 Wafer 最後一筆前有 >1 秒 Gap",
            "處理": "保留並記錄；Feature Engineering 時特別處理時間型特徵",
        },
        {
            "項目": "Process Timestamp Metadata",
            "發現": "96 個 groups 的 metadata 一致，但和 2024 實驗日期不一致，不能當 absolute calendar time",
            "處理": "只作為 Wafer 內部 numeric process time axis 使用",
        },
    ]
)

data_quality_decisions


,項目,發現,處理
0,postox_thickness_nan,157 個原始 fitting-failure NaN，分布在 20 片 Wafer,保留原始欄位並建立 postox_was_interpolated Flag
1,postox_thickness,原作者已對 fitting-failure positions 使用 IDW reconst...,使用完整欄位，但保留 interpolation provenance
2,preox_thickness,89-point grid 中只有 15 個直接量測點，其餘由資料提供者 IDW 重建,保留；Spatial / oxide_etch 解讀必須註明 provenance 限制
3,Duplicate Measurement,0 筆 full-row duplicate；0 筆 Wafer+XY key duplicate,不需去重
4,Spatial Grid,88 片 Wafer 均使用相同 89 個 unique XY positions,可在共同 grid 上進行 Wafer Spatial Analysis
5,Derived Etch Columns,oxide_etch 與 si_etch 公式驗證一致，並繼承組成量測欄位的 reconst...,可作為 Outcome；保留 provenance，且避免把可直接計算 target 的 m...
6,Lot Metadata,日期、空格與 conditioning 格式已標準化，experiment_key date...,使用標準化欄位；未知 conditioning 格式直接停止
7,Process Missing / Inf,31 個共同 Process Features 中 NaN = 0、Inf = 0,不需補值
8,Constant Process Features,31 個共同欄位中有 4 個全域 constant,後續分析排除，Raw Data 保留
9,Process Terminal Gap,69 / 96 Wafer 最後一筆前有 >1 秒 Gap,保留並記錄；Feature Engineering 時特別處理時間型特徵


**這個結果代表什麼？**

Decision Table 把不同性質的問題分開處理。

幾個最重要的決定是：

- interpolation：**完整值保留，但 interpolation flag 也保留**
- constant Process signals：**Raw Data 保留，後續分析排除**
- terminal gap：**不當成中段 missing，不亂刪 sample，但後續時間型 feature 要特別處理**
- timestamp metadata：**只把 `times` 當 Wafer 內 numeric time axis，不當真實實驗日期**

這樣做的好處是，每個問題都有對應處理方式，不會用一個「全部刪掉」的規則硬套所有情況。


### 7.3 把整理好的資料和 QC 規則輸出到 `data/interim/`

前面已經把 Measurement、Lot metadata 和 Process QC 整理完成，接下來把後面會重複用到的結果存下來：

- `measurement_qc.csv`
- `lot_summary_clean.csv`
- `process_qc_summary.csv`
- `usable_process_features.csv`
- `data_quality_decisions.csv`

之後分析 Measurement 和 Lot 資訊時，可以直接讀這些整理好的檔案，不需要每次都從頭清理。

如果之後要看完整的 Process time series，還是要回到 Raw NetCDF，因為 CSV 不會取代原始時間序列。不過後面會沿用這裡已經確認好的 **27 個 usable features、timestamp 的使用方式，以及 terminal gap 的 QC 規則**。

這樣後面的分析會建立在同一套資料處理規則上，不會每一個檔案都自己用不同方式整理。

In [34]:
INTERIM_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

measurement_interim.to_csv(
    INTERIM_DATA_DIR
    / "measurement_qc.csv",
    index=False,
)

lot_summary_clean.to_csv(
    INTERIM_DATA_DIR
    / "lot_summary_clean.csv",
    index=False,
)

process_qc_df.to_csv(
    INTERIM_DATA_DIR
    / "process_qc_summary.csv",
    index=False,
)

pd.DataFrame(
    {
        "feature": usable_process_features
    }
).to_csv(
    INTERIM_DATA_DIR
    / "usable_process_features.csv",
    index=False,
)

data_quality_decisions.to_csv(
    INTERIM_DATA_DIR
    / "data_quality_decisions.csv",
    index=False,
)

print("Interim 資料輸出完成")

Interim 資料輸出完成


**這個結果代表什麼？**

顯示「Interim 資料輸出完成」代表前面整理好的資料和 QC 表已經成功寫到 `data/interim/`。

後面的使用方式會是：

- Measurement / Lot metadata：優先讀 interim 檔
- 完整 Process traces：仍然讀 Raw NetCDF
- Process 分析：沿用這裡確認的 usable-feature list 和 QC 規則

也就是保留完整 time-series 資訊，同時避免重複清理。


### 7.4 先確認 5 個預期輸出檔真的都有寫出來

先做最簡單的 file existence check，確定沒有漏掉任何一個後面需要用的檔案。


In [35]:
interim_files = [
    "measurement_qc.csv",
    "lot_summary_clean.csv",
    "process_qc_summary.csv",
    "usable_process_features.csv",
    "data_quality_decisions.csv",
]

for filename in interim_files:
    path = INTERIM_DATA_DIR / filename

    print(
        f"{filename:<35}",
        "存在" if path.exists() else "找不到",
    )

measurement_qc.csv                  存在
lot_summary_clean.csv               存在
process_qc_summary.csv              存在
usable_process_features.csv         存在
data_quality_decisions.csv          存在


**這個結果代表什麼？**

5 個預期的 interim files 全部顯示「存在」。

所以輸出流程沒有漏檔，後面的 Notebook 需要的 Measurement、Lot、Process QC、usable features 和 decision table 都已經寫出來。


### 7.5 不只看檔案存在，再重新讀回來驗證內容尺寸

檔案存在不代表內容一定寫對。

所以這裡真的從硬碟重新 `read_csv`，確認：

- Measurement row / column count
- Lot row count
- Process QC row count
- usable feature 數量
- Data Quality Decision row count

都和記憶體裡預期的一樣。


In [36]:
# 重新讀取剛剛輸出的 Interim Files

measurement_check = pd.read_csv(
    INTERIM_DATA_DIR / "measurement_qc.csv"
)

lot_check = pd.read_csv(
    INTERIM_DATA_DIR / "lot_summary_clean.csv"
)

process_qc_check = pd.read_csv(
    INTERIM_DATA_DIR / "process_qc_summary.csv"
)

usable_features_check = pd.read_csv(
    INTERIM_DATA_DIR / "usable_process_features.csv"
)

decision_check = pd.read_csv(
    INTERIM_DATA_DIR / "data_quality_decisions.csv"
)


final_output_checks = pd.DataFrame(
    {
        "檢查項目": [
            "Measurement rows = 7832",
            "Measurement columns = 20",
            "Lot rows = 10",
            "Process QC rows = 96",
            "Usable Process Features = 27",
            "Data Quality Decision rows = 11",
        ],
        "是否通過": [
            measurement_check.shape[0] == 7832,
            measurement_check.shape[1] == 20,
            lot_check.shape[0] == 10,
            process_qc_check.shape[0] == 96,
            usable_features_check.shape[0] == 27,
            decision_check.shape[0] == 11,
        ],
    }
)

final_output_checks

,檢查項目,是否通過
0,Measurement rows = 7832,True
1,Measurement columns = 20,True
2,Lot rows = 10,True
3,Process QC rows = 96,True
4,Usable Process Features = 27,True
5,Data Quality Decision rows = 11,True


**這個結果代表什麼？**

6 個 read-back checks 全部是 `True`：

- Measurement rows = **7,832**
- Measurement columns = **20**
- Lot rows = **10**
- Process QC rows = **96**
- Usable Process Features = **27**
- Data Quality Decision rows = **11**

所以檔案不只是「有寫出去」，內容的主要尺寸也沒有在輸出過程中改掉。


最後再用 `assert` 把剛才的輸出驗證變成硬性條件。

這樣如果未來上游程式改動，造成任何一個輸出尺寸不符合預期，01 就會直接停在這裡，不把錯的檔案交給 02。


In [37]:
assert final_output_checks["是否通過"].all(), (
    "Interim output validation failed."
)

print("所有 Interim Output 驗證通過。")

所有 Interim Output 驗證通過。


**這個結果代表什麼？**

顯示「所有 Interim Output 驗證通過」，代表前面寫出的 5 個檔案都通過存在性和關鍵尺寸檢查。

所以 01 的輸出和這次 Data Quality 結論是對得上的，可以交給 02 接著做 Wafer Quality Analysis。


## 8. 這份 Notebook 的整體結論

01 做完之後，可以確認這份資料不是完全沒有品質問題，但主要問題都已經找出來，而且每一種問題都有對應的處理方式，不需要看到缺失就大量刪資料，也不需要自己再做一次不必要的補值。

Measurement 方面，最主要的缺失是在 `postox_thickness_nan`：總共有 **157 個 fitting-failure NaN，分布在 20 片 Wafer**，而且 Excel 的紀錄和 CSV 逐片完全對得上。這些位置資料提供者已經在完整的 `postox_thickness` 裡做過重建，所以我不再補第二次，而是保留完整數值，再用 `postox_was_interpolated` 把這 157 個位置標記起來。這樣後面還是可以分析，但看到這些位置時也知道它們不是原本成功 fitting 出來的值。

空間資料本身沒有發現結構問題：**沒有重複列、沒有重複 Wafer+XY key，88 片 Wafer 都使用同一組 89 個 XY positions，而且所有量測點都在 200 mm Wafer 的範圍內。** `oxide_etch` 和 `si_etch` 的公式也和原始欄位對得上，所以後面可以拿它們當品質指標使用，只是仍然要記得它們會繼承原始量測和重建值的不確定性。

Lot 資訊整理完後，10 個 Lot 的日期、Wafer 數、Conditioning count 和 surface 都可以正常使用，而且 `experiment_key` 裡的日期和 Lot 日期沒有任何一筆對不上。

Process Data 方面，96 片 Wafer 的 31 個共同 Features 沒有 NaN 或 Inf，但其中有 **4 個欄位全域固定為 0**，所以後面只保留 **27 個真的有變化的 Process signals**。時間軸的中位取樣間隔全部是 **0.2 秒**，符合大約 5 Hz；不過 **69 / 96 片 Wafer** 在最後一個 sample 前有一個 >1 秒的 gap。因為這些 gap 全部集中在序列尾端，所以我不把它們當成製程中段缺資料，而是保留 QC 紀錄，提醒後面在算 duration、slope 或 time-weighted feature 時不要讓最後這個 gap 影響結果。

還有一個很重要的限制：NetCDF 的 timestamp metadata 雖然 **96 / 96 都能正常解碼**，但解碼後的日期跟 Group Name 裡真正的 2024 實驗日期是 **0 / 96 相符**。所以後面不能把 `times` 當成真實日期時間，只能把它當成每片 Wafer 裡面的數值時間軸；真正的實驗日期要用 Group Name 或 Lot 資訊。

最後，`measurement_qc.csv`、`lot_summary_clean.csv`、`process_qc_summary.csv`、`usable_process_features.csv` 和 `data_quality_decisions.csv` 都已經重新讀回，尺寸和內容檢查也全部通過。

所以 01 真正完成的事情不是把資料「洗到看起來完全沒問題」，而是把**哪些資料有問題、問題在哪裡、後面要怎麼用**都先交代清楚。接下來 02 就可以直接用這些整理好的 Measurement 資料，開始分析 Wafer 的 Quality 分布、Lot 差異和空間 pattern。

## 9. 下一步

下一份是：

**`02_wafer_quality_analysis.ipynb`｜Wafer Quality Analysis**

01 已經把「資料能不能用、限制在哪裡」處理完。02 就不再重做清理，而是直接用 `measurement_qc.csv` 開始看：

> **Si Etch 品質在 Wafer、Lot、wafer sequence 和 spatial pattern 上到底怎麼變。**

這樣分析順序會比較乾淨：先把資料品質處理好，再去解讀品質差異。
